In [0]:
# COMMAND ----------

from pyspark.sql import functions as F


# ============================================================
# 1. CONFIGURATION
# ============================================================

CATALOG = "aml_engine"
SCHEMA = "aml_poc"

BRONZE_TABLE = f"{CATALOG}.{SCHEMA}.bronze_transactions"
SILVER_TABLE = f"{CATALOG}.{SCHEMA}.silver_transactions"

S3_BASE_PATH = "s3://zubair-s3-demo/raw_dataset/aml"
S3_DELTA_PATH = f"{S3_BASE_PATH}/delta_tables"

# Existing Silver Delta table location
SILVER_TRANSACTIONS_PATH = (
    f"{S3_DELTA_PATH}/silver_transactions"
)

# IMPORTANT:
# This checkpoint must remain permanently.
# Do NOT delete it between runs.
SILVER_CHECKPOINT_PATH = (
    f"{S3_BASE_PATH}/checkpoints/silver_transactions"
)


# ============================================================
# 2. READ NEW DATA FROM BRONZE INCREMENTALLY
# ============================================================

bronze_transactions_df = (
    spark.readStream
        .format("delta")
        .table(BRONZE_TABLE)
)

print(f"Bronze source : {BRONZE_TABLE}")
print("Reading Bronze incrementally using Structured Streaming.")


# ============================================================
# 3. TRANSFORM BRONZE -> SILVER
# ============================================================

silver_transactions_df = (
    bronze_transactions_df

    .select(
        # ----------------------------------------------------
        # Transaction ID
        # ----------------------------------------------------
        F.col("TX_ID")
            .cast("long")
            .alias("tx_id"),

        # ----------------------------------------------------
        # Sender account
        # ----------------------------------------------------
        F.col("SENDER_ACCOUNT_ID")
            .cast("long")
            .alias("sender_account_id"),

        # ----------------------------------------------------
        # Receiver account
        # ----------------------------------------------------
        F.col("RECEIVER_ACCOUNT_ID")
            .cast("long")
            .alias("receiver_account_id"),

        # ----------------------------------------------------
        # Transaction type
        # ----------------------------------------------------
        F.upper(
            F.trim(F.col("TX_TYPE"))
        ).alias("tx_type"),

        # ----------------------------------------------------
        # Transaction amount
        # ----------------------------------------------------
        F.col("TX_AMOUNT")
            .cast("double")
            .alias("tx_amount"),

        # ----------------------------------------------------
        # Event time
        # Your current dataset contains values such as 0,1,2...
        # ----------------------------------------------------
        F.col("TIMESTAMP")
            .cast("long")
            .alias("event_time"),

        # ----------------------------------------------------
        # Fraud label
        #
        # Historical data:
        #     True / False
        #
        # New operational data:
        #     NULL
        #
        # Do NOT make this field mandatory.
        # ----------------------------------------------------
        F.col("IS_FRAUD")
            .cast("boolean")
            .alias("is_fraud"),

        # ----------------------------------------------------
        # Alert ID
        # ----------------------------------------------------
        F.col("ALERT_ID")
            .cast("long")
            .alias("alert_id"),

        # ----------------------------------------------------
        # Ingestion / schema metadata
        # ----------------------------------------------------
        F.col("_rescued_data"),
        F.col("_ingested_timestamp"),
        F.col("_source_file")
    )
)


# ============================================================
# 4. DATA QUALITY FILTERS
# ============================================================
#
# IMPORTANT:
# IS_FRAUD is intentionally NOT validated here.
#
# Historical:
#     is_fraud = True / False
#
# New:
#     is_fraud = NULL
#
# Both are valid transaction records.
# ============================================================

silver_transactions_df = (
    silver_transactions_df

    # Transaction ID is mandatory
    .filter(
        F.col("tx_id").isNotNull()
    )

    # Sender must exist
    .filter(
        F.col("sender_account_id").isNotNull()
    )

    # Receiver must exist
    .filter(
        F.col("receiver_account_id").isNotNull()
    )

    # Transaction amount must exist
    .filter(
        F.col("tx_amount").isNotNull()
    )

    # Transaction amount must be positive
    .filter(
        F.col("tx_amount") > 0
    )
)


# ============================================================
# 5. ADD SILVER PROCESSING TIMESTAMP
# ============================================================

silver_transactions_df = (
    silver_transactions_df
    .withColumn(
        "_silver_processed_timestamp",
        F.current_timestamp()
    )
)


# ============================================================
# 6. WRITE INCREMENTALLY TO EXISTING SILVER DELTA TABLE
# ============================================================
#
# append mode is used because transactions are append-oriented.
#
# availableNow=True means:
#   - process all currently available Bronze records
#   - stop when caught up
#
# The checkpoint ensures previously processed Bronze data
# is not processed again on the next run.
# ============================================================

query = (
    silver_transactions_df
        .writeStream
        .format("delta")
        .outputMode("append")

        .option(
            "checkpointLocation",
            SILVER_CHECKPOINT_PATH
        )

        .option(
            "path",
            SILVER_TRANSACTIONS_PATH
        )

        .trigger(
            availableNow=True
        )

        .toTable(
            SILVER_TABLE
        )
)


# ============================================================
# 7. WAIT FOR COMPLETION
# ============================================================

query.awaitTermination()

print("Silver Transactions incremental processing completed.")
print(f"Unity Catalog table : {SILVER_TABLE}")
print(f"S3 location         : {SILVER_TRANSACTIONS_PATH}")
print(f"Checkpoint location  : {SILVER_CHECKPOINT_PATH}")